# Guía de bolsillo: SCD Tipo 1, 2 y 3 (dimensiones con historial)

**Para usar en la capa Gold.** Ejemplo genérico de e-commerce con datos cargados a mano, así no depende de ningún dataset.

**Cómo usarlo**
- Corré el notebook de arriba hacia abajo (Run all). Se puede re-ejecutar entero: arranca con `CREATE OR REPLACE`.
- Todo vive en un schema de práctica: `bootcamp.demo_scd`. Para otro catálogo o schema, cambiá solo las dos primeras líneas de la sección 4.
- Las celdas de las secciones 5, 6 y 7 son **plantillas**: cambiás nombres de tabla y de columnas y las llevás a Gold.

**Contenido**
1. La idea en 30 segundos
2. Comparación lado a lado (tabla + el mismo cliente en los 3 modelos)
3. Cómo modelar una dimensión con historial (SK, BK, fechas, hash) y cómo se conecta la fact
4. Datos de ejemplo
5. SCD Tipo 1 (pisar)
6. SCD Tipo 2 (versionar con filas)
7. SCD Tipo 3 (valor actual + anterior)
8. Comparación con la fact: ¿qué cambia en los reportes?
9. Errores clásicos y checklist
10. Otros tipos y atajos

---
# 1. La idea en 30 segundos

Una dimensión describe cosas que **cambian con el tiempo**: un cliente se muda, un producto cambia de categoría, un cliente pasa de Standard a Premium.

La única pregunta que define el tipo de SCD (*Slowly Changing Dimension*) es:

> **Cuando cambia un atributo, ¿qué hago con el valor viejo?**

| Tipo | Respuesta | En una frase |
|---|---|---|
| **Tipo 1** | Lo piso | "El pasado no me importa" (o era un error). |
| **Tipo 2** | Dejo la fila vieja y creo una **fila nueva** | "Necesito saber cómo era en cada momento". |
| **Tipo 3** | Guardo el viejo en una **columna nueva** (`*_anterior`) | "Solo necesito el antes y el después del último cambio". |

**Los tres guardan (o no) historial de la dimensión, pero cada uno guarda una cantidad distinta:**
- Tipo 1: **nada**.
- Tipo 2: **todo** (todas las versiones).
- Tipo 3: **solo el valor inmediatamente anterior**, de un atributo.

---
# 2. Comparación lado a lado

## 2.1 Tabla resumen

| | **Tipo 1: pisar** | **Tipo 2: nueva fila** | **Tipo 3: nueva columna** |
|---|---|---|---|
| **Qué hace al cambiar un atributo** | `UPDATE` sobre la misma fila | Cierra la fila vigente (`fecha_hasta`, `es_actual = FALSE`) e **inserta** una fila nueva | `UPDATE` sobre la misma fila, moviendo el valor actual a `*_anterior` |
| **Historial que conserva** | Ninguno | Completo | Un solo cambio hacia atrás, solo del atributo elegido |
| **Filas por cliente (BK)** | 1 | 1 **por versión** | 1 |
| **Surrogate Key (SK)** | 1 por cliente, no cambia | **Una nueva por versión** | 1 por cliente, no cambia |
| **Columnas extra** | Ninguna (a lo sumo `hash_diff`) | `fecha_desde`, `fecha_hasta`, `es_actual`, `hash_diff` | `<atributo>_anterior`, `fecha_cambio_<atributo>` |
| **¿Puedo responder "cómo era el 15 de febrero"?** | No | **Sí** | No |
| **Un reporte viejo, ¿cambia si el cliente cambia?** | Sí: se reescribe con el valor nuevo | **No**: queda con el valor de su época | Sí: se reescribe (la columna actual manda) |
| **Cómo se une la fact** | Por SK | Por la SK **de la versión vigente al momento del evento** | Por SK |
| **Crece en...** | No crece | Filas | Columnas (una por atributo que quiero recordar) |
| **Complejidad** | Baja | Alta | Media |
| **Ejemplo e-commerce** | Corregir el typo de un nombre o email | Cliente que se muda de provincia; producto que cambia de categoría | Comparar categoría vieja vs nueva tras una reorganización |

## 2.2 El mismo cliente en los tres modelos

**Historia de Ana:** vivía en **Rosario** (enero) → se mudó a **Córdoba** (marzo) → se mudó a **Mendoza** (junio).

**Tipo 1** (1 fila, sin historial):

| cliente_sk | cliente_id | nombre | ciudad |
|---|---|---|---|
| 1 | 1 | Ana Pérez | Mendoza |

**Tipo 2** (1 fila por versión):

| cliente_sk | cliente_id | nombre | ciudad | fecha_desde | fecha_hasta | es_actual |
|---|---|---|---|---|---|---|
| 1 | 1 | Ana Pérez | Rosario | 2026-01-01 | 2026-03-01 | FALSE |
| 5 | 1 | Ana Pérez | Córdoba | 2026-03-01 | 2026-06-01 | FALSE |
| 7 | 1 | Ana Pérez | Mendoza | 2026-06-01 | 9999-12-31 | TRUE |

**Tipo 3** (1 fila, valor actual + anterior). Rosario se perdió:

| cliente_sk | cliente_id | nombre | ciudad | ciudad_anterior | fecha_cambio_ciudad |
|---|---|---|---|---|---|
| 1 | 1 | Ana Pérez | Mendoza | Córdoba | 2026-06-01 |

**Prueba práctica:** una venta de Ana del 10 de febrero, ¿en qué ciudad se hizo?
- Tipo 1: dice **Mendoza** (incorrecto: en febrero vivía en Rosario).
- Tipo 2: dice **Rosario** (correcto).
- Tipo 3: dice **Mendoza**. Y lo único que sabe del pasado es "antes fue Córdoba", que tampoco responde la pregunta.

## 2.3 Cómo elegir (árbol de decisión)

1. **¿Es la corrección de un error?** (typo, dato mal cargado) → **Tipo 1**. No tiene sentido guardar un error.
2. **¿Alguien necesita saber qué valor tenía el atributo en el momento del evento?** (ventas por la provincia que tenía el cliente cuando compró) → **Tipo 2**.
3. **¿Solo necesitan comparar contra el valor inmediatamente anterior?** (migración de segmento: antes vs después) → **Tipo 3**.
4. **¿Nadie lo pidió?** → **Tipo 1** (es el valor por defecto; la mayoría de las dimensiones son Tipo 1).

> Regla de oro: **SCD2 solo cuando se cumplen las tres**: el atributo realmente cambia, alguien necesita el histórico y el costo de mantenerlo vale la pena. Una dimensión con 9 valores que casi no cambian no necesita SCD2.

---
# 3. Cómo modelar una dimensión con historial

## 3.1 Piezas de una dimensión

| Pieza | Qué es | Ejemplo |
|---|---|---|
| **Business Key (BK)** | El identificador del negocio. Identifica a la **entidad**. Se usa para el match en el `MERGE`. | `cliente_id` |
| **Surrogate Key (SK)** | Clave artificial del DW (`GENERATED ALWAYS AS IDENTITY`). Identifica a la **fila (versión)**. La fact apunta a esta. | `cliente_sk` |
| **Atributos** | Lo descriptivo que puede cambiar. | `ciudad`, `provincia`, `segmento` |
| **`hash_diff`** | Hash (SHA-256) de los atributos que me interesa vigilar. Si el hash cambió, algo cambió. Evita comparar columna por columna y los problemas con `NULL`. | `sha2(concat_ws(...), 256)` |
| **`fecha_desde` / `fecha_hasta`** | Intervalo de vigencia de la versión (solo Tipo 2). La versión vigente tiene `fecha_hasta = 9999-12-31`. | `2026-03-01` a `2026-06-01` |
| **`es_actual`** | Bandera para filtrar rápido la versión vigente (solo Tipo 2). | `TRUE` |

**En Tipo 2, una BK tiene muchas SK** (una por versión). El SK identifica la versión; el BK identifica al cliente.

## 3.2 Reglas de diseño para el Tipo 2

1. **Intervalo semiabierto `[fecha_desde, fecha_hasta)`**: la fecha desde se incluye y la fecha hasta no. La versión vieja termina el mismo día que empieza la nueva (`fecha_hasta` de una = `fecha_desde` de la siguiente) y no hay solapamientos ni huecos. Se consulta con `fecha >= fecha_desde AND fecha < fecha_hasta`, **no con `BETWEEN`** (el día del cambio caería en las dos versiones).
2. **`fecha_desde` = fecha efectiva del cambio**, no necesariamente la fecha en que corrió el pipeline. Si la fuente trae la fecha del cambio o la fecha del snapshot, usá esa. `CURRENT_DATE()` solo sirve si cargás todos los días y no hacés recargas históricas. En este notebook uso `snapshot_fecha`.
3. **`hash_diff` solo con atributos que importan para el historial.** Si metés una columna que cambia todo el tiempo (`ultimo_login`, `updated_at`) generás una versión nueva por cada carga.
4. **Cargar en orden cronológico.** Si procesás el snapshot de junio antes que el de marzo, el historial queda mal.
5. **Una sola fila vigente por BK.** Se verifica con una query (sección 6).

## 3.3 Cómo se conecta la fact (lo más importante para Gold)

La fact **guarda la SK de la versión que estaba vigente cuando ocurrió el evento**. Se resuelve **al cargar la fact**, con un lookup por BK y fecha:

```sql
JOIN dim_cliente_scd2 d
  ON  d.cliente_id  = v.cliente_id          -- BK
  AND v.fecha_venta >= d.fecha_desde        -- fecha del evento dentro del intervalo
  AND v.fecha_venta <  d.fecha_hasta
```

Después los reportes unen por SK (`f.cliente_sk = d.cliente_sk`) y no hace falta mirar fechas: cada venta ya "sabe" con qué versión del cliente ocurrió.

- **Tipo 1 y Tipo 3:** el SK no cambia nunca, así que el lookup es solo por BK (sin fechas).
- **Tipo 2:** si unís la fact por BK sin mirar fechas, cada venta se **duplica** tantas veces como versiones tenga el cliente (el error clásico, lo vemos en la sección 8).

---
# 4. Datos de ejemplo (ingesta manual)

Simulamos 3 **snapshots** de la tabla de clientes de origen (como si el sistema transaccional nos mandara la foto completa cada tanto):

| Snapshot | Qué pasa |
|---|---|
| **2026-01-01** | Carga inicial: Ana (Rosario), Bruno (Córdoba), Carla (CABA). |
| **2026-03-01** | Ana se muda a **Córdoba**. Aparece Diego (cliente nuevo, Mendoza). |
| **2026-06-01** | Ana se muda a **Mendoza**. Carla pasa de **Standard a Premium**. |

Más una tabla de ventas con la que después probamos los reportes.

In [0]:
%sql
-- Catálogo y schema de práctica (cambiá estas dos líneas si querés otro lugar)
USE CATALOG bootcamp;
CREATE SCHEMA IF NOT EXISTS demo_scd;
USE SCHEMA demo_scd;

In [0]:
%sql
-- Origen simulado: 3 fotos (snapshots) de la tabla de clientes
CREATE OR REPLACE TABLE raw_clientes_snapshots AS
SELECT * FROM VALUES
  -- Snapshot 1: carga inicial
  (DATE'2026-01-01', 1, 'Ana Pérez',    'ana@mail.com',    'Rosario',  'Santa Fe',     'Standard'),
  (DATE'2026-01-01', 2, 'Bruno Díaz',   'bruno@mail.com',  'Córdoba',  'Córdoba',      'Standard'),
  (DATE'2026-01-01', 3, 'Carla Gómez',  'carla@mail.com',  'CABA',     'Buenos Aires', 'Standard'),
  -- Snapshot 2: Ana se muda a Córdoba; llega Diego (nuevo)
  (DATE'2026-03-01', 1, 'Ana Pérez',    'ana@mail.com',    'Córdoba',  'Córdoba',      'Standard'),
  (DATE'2026-03-01', 2, 'Bruno Díaz',   'bruno@mail.com',  'Córdoba',  'Córdoba',      'Standard'),
  (DATE'2026-03-01', 3, 'Carla Gómez',  'carla@mail.com',  'CABA',     'Buenos Aires', 'Standard'),
  (DATE'2026-03-01', 4, 'Diego Ruiz',   'diego@mail.com',  'Mendoza',  'Mendoza',      'Standard'),
  -- Snapshot 3: Ana se muda a Mendoza; Carla pasa a Premium
  (DATE'2026-06-01', 1, 'Ana Pérez',    'ana@mail.com',    'Mendoza',  'Mendoza',      'Standard'),
  (DATE'2026-06-01', 2, 'Bruno Díaz',   'bruno@mail.com',  'Córdoba',  'Córdoba',      'Standard'),
  (DATE'2026-06-01', 3, 'Carla Gómez',  'carla@mail.com',  'CABA',     'Buenos Aires', 'Premium'),
  (DATE'2026-06-01', 4, 'Diego Ruiz',   'diego@mail.com',  'Mendoza',  'Mendoza',      'Standard')
AS t(snapshot_fecha, cliente_id, nombre, email, ciudad, provincia, segmento);

SELECT * FROM raw_clientes_snapshots ORDER BY snapshot_fecha, cliente_id;

In [0]:
%sql
-- Vista con el hash_diff ya calculado (se usa en los 3 tipos).
-- COALESCE evita que un NULL "desaparezca" del hash (concat_ws ignora NULLs).
CREATE OR REPLACE TEMP VIEW v_clientes_hash AS
SELECT
  snapshot_fecha, cliente_id, nombre, email, ciudad, provincia, segmento,
  sha2(concat_ws('||',
        COALESCE(nombre,    '~'),
        COALESCE(email,     '~'),
        COALESCE(ciudad,    '~'),
        COALESCE(provincia, '~'),
        COALESCE(segmento,  '~')), 256) AS hash_diff
FROM raw_clientes_snapshots;

SELECT * FROM v_clientes_hash WHERE cliente_id = 1 ORDER BY snapshot_fecha;

## Qué columnas incluir en el `hash_diff` según el tipo de SCD

- **Tipo 1:** el hash suele cubrir **todos los atributos que querés mantener sincronizados**, porque cualquier cambio debe pisar el valor.
- **Tipo 2:** el hash cubre **solo los atributos que justifican una versión nueva**. Si el `email` cambia y no te importa su historial, no lo incluyas, o vas a generar una versión por cada cambio de email.
- **Híbrido (algunos atributos Tipo 2 y otros Tipo 1):** se usan **dos hashes**. `hash_tipo2` decide si se crea una versión nueva, y `hash_tipo1` decide si hay que pisar los atributos Tipo 1 en las versiones existentes.

> En el notebook de la guía armé el hash con todos los atributos (`nombre`, `email`, `ciudad`, `provincia`, `segmento`) por simplicidad. En un caso real, revisá qué atributos justifican una versión nueva y sacá del hash los demás.

## Cuidados al construirlo

- **`COALESCE` en cada columna.** `concat_ws` ignora los `NULL`, así que `('a', NULL, 'b')` y `('a', 'b', NULL)` generarían el mismo hash.
- **Separador** (`'||'`) entre columnas, para que `('ab', 'c')` y `('a', 'bc')` no den igual.
- **Normalización:** si `'Córdoba '` y `'Córdoba'` deben considerarse iguales, aplicá `TRIM` (y `UPPER` si corresponde) antes de hashear. De lo contrario un espacio de más genera una versión nueva.
- **Orden de columnas fijo.** Si cambiás el orden o la lista de columnas del hash, **todas las filas van a parecer modificadas** en la próxima carga. Cambiar su definición implica recalcular el hash de lo que ya está cargado.
- **Formato de los tipos numéricos y fechas.** Hashear `10.0` y `10.00` como texto da distinto. Casteá a un formato consistente.

In [0]:
%sql
-- Ventas de ejemplo (BK del cliente + fecha del evento + monto)
CREATE OR REPLACE TABLE raw_ventas AS
SELECT * FROM VALUES
  (100, 1, DATE'2026-02-10', 1000.00),  -- Ana, vivía en Rosario
  (101, 1, DATE'2026-04-15', 2000.00),  -- Ana, vivía en Córdoba
  (102, 1, DATE'2026-07-01', 3000.00),  -- Ana, vive en Mendoza
  (103, 2, DATE'2026-02-20',  500.00),  -- Bruno
  (104, 3, DATE'2026-05-05',  800.00),  -- Carla, era Standard
  (105, 3, DATE'2026-07-10', 1200.00),  -- Carla, ya es Premium
  (106, 4, DATE'2026-04-01',  700.00)   -- Diego
AS t(venta_id, cliente_id, fecha_venta, monto);

SELECT COUNT(*) AS ventas, SUM(monto) AS total FROM raw_ventas;   -- 7 ventas, total 9200

---
# 5. SCD Tipo 1: pisar el valor

**Cuándo:** correcciones de errores, o atributos donde el pasado no importa (o se quiere que los reportes viejos muestren el valor nuevo).

**Cómo se modela:** la dimensión es la más simple. Solo BK + SK + atributos (+ `hash_diff` para detectar cambios).

**Cómo se carga:** `MERGE` por BK.
- `WHEN MATCHED` y el hash cambió: `UPDATE` de los atributos.
- `WHEN NOT MATCHED`: `INSERT`.

In [0]:
%sql
CREATE OR REPLACE TABLE dim_cliente_scd1 (
  cliente_sk  BIGINT GENERATED ALWAYS AS IDENTITY,  -- SK (la que usa la fact)
  cliente_id  INT NOT NULL,                         -- BK
  nombre      STRING,
  email       STRING,
  ciudad      STRING,
  provincia   STRING,
  segmento    STRING,
  hash_diff   STRING
) USING DELTA
COMMENT 'Dimension cliente - SCD Tipo 1 (pisa el valor, sin historial)';

In [0]:
%sql
select * from dim_cliente_scd1;

In [0]:
%sql
-- ===== Carga del snapshot 1 (inicial) =====
CREATE OR REPLACE TEMP VIEW stg_clientes AS
SELECT * FROM v_clientes_hash WHERE snapshot_fecha = DATE'2026-01-01';

MERGE INTO dim_cliente_scd1 AS t
USING stg_clientes AS s
  ON t.cliente_id = s.cliente_id
WHEN MATCHED AND t.hash_diff <> s.hash_diff THEN
  UPDATE SET nombre = s.nombre, email = s.email, ciudad = s.ciudad,
             provincia = s.provincia, segmento = s.segmento, hash_diff = s.hash_diff
WHEN NOT MATCHED THEN
  INSERT (cliente_id, nombre, email, ciudad, provincia, segmento, hash_diff)
  VALUES (s.cliente_id, s.nombre, s.email, s.ciudad, s.provincia, s.segmento, s.hash_diff);

SELECT cliente_sk, cliente_id, nombre, ciudad, provincia, segmento
FROM dim_cliente_scd1 ORDER BY cliente_id;

In [0]:
%sql
-- ===== Carga del snapshot 2 (Ana se muda a Córdoba; entra Diego) =====
CREATE OR REPLACE TEMP VIEW stg_clientes AS
SELECT * FROM v_clientes_hash WHERE snapshot_fecha = DATE'2026-03-01'; --mirar v_clientes_hash para entender por fecha

MERGE INTO dim_cliente_scd1 AS t --target
USING stg_clientes AS s --source
  ON t.cliente_id = s.cliente_id
WHEN MATCHED AND t.hash_diff <> s.hash_diff THEN 
--cambia el hash para An cuando se muda a Córdoba, ya que el hash lo incluye
--el cliente id es para unir target y source y el hash es lo que MARCA la diferencia
  UPDATE SET 
  nombre = s.nombre, 
  email = s.email, 
  ciudad = s.ciudad,
  provincia = s.provincia, 
  segmento = s.segmento, 
  hash_diff = s.hash_diff
WHEN NOT MATCHED THEN --se inserta Diego, porque no estaba antes
  INSERT (cliente_id, nombre, email, ciudad, provincia, segmento, hash_diff) --los de target
  VALUES (s.cliente_id, s.nombre, s.email, s.ciudad, s.provincia, s.segmento, s.hash_diff); --los de source

-- Ana sigue siendo 1 sola fila, con el MISMO cliente_sk; Rosario ya no existe en ningún lado
SELECT cliente_sk, cliente_id, nombre, ciudad, provincia, segmento
FROM dim_cliente_scd1 ORDER BY cliente_id;

In [0]:
%sql
-- ===== Carga del snapshot 3 (Ana a Mendoza; Carla pasa a Premium) =====
CREATE OR REPLACE TEMP VIEW stg_clientes AS
SELECT * FROM v_clientes_hash WHERE snapshot_fecha = DATE'2026-06-01';

MERGE INTO dim_cliente_scd1 AS t
USING stg_clientes AS s
  ON t.cliente_id = s.cliente_id
WHEN MATCHED AND t.hash_diff <> s.hash_diff THEN
  UPDATE SET nombre = s.nombre, email = s.email, ciudad = s.ciudad,
             provincia = s.provincia, segmento = s.segmento, hash_diff = s.hash_diff
WHEN NOT MATCHED THEN
  INSERT (cliente_id, nombre, email, ciudad, provincia, segmento, hash_diff)
  VALUES (s.cliente_id, s.nombre, s.email, s.ciudad, s.provincia, s.segmento, s.hash_diff);

SELECT cliente_sk, cliente_id, nombre, ciudad, provincia, segmento
FROM dim_cliente_scd1 ORDER BY cliente_id;

--RDO: siempre pisa lo anterior, como si fuera un INSERT OVERWRITE.
--el MERGE es mas óptimo para tablas grandes porque justamente pisa lo que tiene que pisar.
--en cambio el INSERT OVERWRITE, borra todo y carga de nuevo, es menos eficiente

**Qué mirar:**
- 4 filas en total, una por cliente, y cada `cliente_sk` es estable.
- Ana figura en **Mendoza**: no hay rastro de Rosario ni de Córdoba. Carla figura como **Premium** desde siempre, según esta tabla.
- Re-ejecutar una carga con los mismos datos **no cambia nada** (el hash coincide): el `MERGE` es idempotente.

---
# 6. SCD Tipo 2: una fila nueva por cada versión

**Cuándo:** el atributo cambia por un hecho real del negocio y los reportes históricos tienen que mostrar el valor de su época.

**Cómo se modela:** además de BK, SK y atributos, se agregan `hash_diff`, `fecha_desde`, `fecha_hasta` y `es_actual`.

**Cómo se carga:** cada cambio requiere **dos operaciones**: (1) **cerrar** la versión vigente y (2) **insertar** la nueva versión. Un `MERGE` normal no puede hacer las dos sobre la misma fila. El truco es alimentar el `MERGE` con **dos copias** de cada fila que cambió:

| Fila del staging | `merge_key` | Qué pasa en el `MERGE` |
|---|---|---|
| Copia A (todas las filas) | el `cliente_id` | Si matchea una versión vigente y el hash cambió → **cierra** la versión. Si no matchea (cliente nuevo) → **inserta** la primera versión. |
| Copia B (solo las que cambiaron) | `NULL` | Un `NULL` nunca matchea → cae en `NOT MATCHED` → **inserta** la versión nueva. |

Así queda todo en **una sola sentencia atómica** (no puede quedar a medias, por ejemplo con la versión cerrada pero sin la nueva).

**⚠️ Se recomienda ir paso a paso psra entender los cambios y como los guarda SCD2.**

**⚠️ Si uno quiere empezar desde el inicio, dropear la dim_cliente_scd2 y empezar de nuevo**

In [0]:
%sql
DROP table if exists bootcamp.demo_scd.dim_cliente_scd2;

CREATE OR REPLACE TABLE bootcamp.demo_scd.dim_cliente_scd2 (
  cliente_sk   BIGINT GENERATED ALWAYS AS IDENTITY,  -- SK: una por VERSION
  cliente_id   INT NOT NULL,                         -- BK: se repite entre versiones
  nombre       STRING,
  email        STRING,
  ciudad       STRING,
  provincia    STRING,
  segmento     STRING,
  hash_diff    STRING NOT NULL,
  fecha_desde  DATE NOT NULL,                        -- inicio de vigencia (incluida)
  fecha_hasta  DATE NOT NULL,                        -- fin de vigencia (excluida); 9999-12-31 = vigente
  es_actual    BOOLEAN NOT NULL
) USING DELTA
COMMENT 'Dimension cliente - SCD Tipo 2 (historial completo por versiones)';

In [0]:
%sql
-- ===== Carga del snapshot 1 (inicial) =====
CREATE OR REPLACE TEMP VIEW stg_clientes AS
SELECT * FROM v_clientes_hash WHERE snapshot_fecha = DATE'2026-01-01'; --tiene columna hash_diff


SELECT * FROM stg_clientes;

MERGE INTO dim_cliente_scd2 AS t
USING (
  -- Copia A: todas las filas del staging, con su BK como merge_key
  SELECT s.cliente_id AS merge_key, s.* --merge key es cliente_id.
  FROM stg_clientes s -- solo los usuarios en 2026-01-01

  UNION ALL

  -- Copia B: solo las filas que CAMBIARON respecto de la versión vigente, con merge_key NULL (al inicio), con la carga inicial
  SELECT CAST(NULL AS INT) AS merge_key, s.*
  FROM stg_clientes s
  JOIN dim_cliente_scd2 d
    ON d.cliente_id = s.cliente_id AND d.es_actual = TRUE
  WHERE d.hash_diff <> s.hash_diff --esta es la clave para obtener en esta parte los que ya existen en el target, PERO cambiaron su estado o columna que queremos identificar cocmo cambio

) AS src
  ON t.cliente_id = src.merge_key AND t.es_actual = TRUE
WHEN MATCHED AND t.hash_diff <> src.hash_diff THEN             -- cerrar la versión vieja
  UPDATE SET fecha_hasta = src.snapshot_fecha, es_actual = FALSE
WHEN NOT MATCHED THEN                                          -- cliente nuevo O versión nueva
  INSERT (cliente_id, nombre, email, ciudad, provincia, segmento, hash_diff,
          fecha_desde, fecha_hasta, es_actual)
  VALUES (src.cliente_id, src.nombre, src.email, src.ciudad, src.provincia, src.segmento, src.hash_diff,
          src.snapshot_fecha, DATE'9999-12-31', TRUE);

SELECT *
FROM dim_cliente_scd2 ORDER BY cliente_id, fecha_desde;

In [0]:
%sql
select * from dim_cliente_scd2;

In [0]:
%sql
-- ===== Carga del snapshot 2 (Ana se muda a Córdoba; entra Diego) =====
CREATE OR REPLACE TEMP VIEW stg_clientes AS
SELECT * FROM v_clientes_hash WHERE snapshot_fecha = DATE'2026-03-01'; -- 4 datos de cliente, Ana mudada y Diego new

select * from stg_clientes;

MERGE INTO dim_cliente_scd2 AS t 
--USING = (todas las filas del staging, con merge_key = BK)
       --        + (una copia extra de las que cambiaron, con merge_key = NULL)

USING (
  SELECT s.cliente_id AS merge_key, s.*
  FROM stg_clientes s --todo lo del nuevo cambio ('2026-03-01')
  UNION ALL
  SELECT CAST(NULL AS INT) AS merge_key, s.* --obteno los datos DE OSURCE, NO MEZCLO CON NADA DE TARGET
  FROM stg_clientes s --todo esto, no lo de dim_clientescd2
  JOIN dim_cliente_scd2 d
    ON d.cliente_id = s.cliente_id AND d.es_actual = TRUE
  WHERE d.hash_diff <> s.hash_diff
) AS src
  ON t.cliente_id = src.merge_key AND t.es_actual = TRUE
WHEN MATCHED AND t.hash_diff <> src.hash_diff THEN
  UPDATE SET 
  fecha_hasta = src.snapshot_fecha, --El registro de Ana en Rosario, queda obsoleto 
  es_actual = FALSE
WHEN NOT MATCHED THEN
  INSERT (cliente_id, nombre, email, ciudad, provincia, segmento, hash_diff,
          fecha_desde, fecha_hasta, es_actual)
  VALUES (src.cliente_id, src.nombre, src.email, src.ciudad, 
          src.provincia, src.segmento, src.hash_diff,
          src.snapshot_fecha, 
          DATE'9999-12-31', --esto es la fecha_hasta
          TRUE); --TRUE es el campo es_actual

-- Ana: 2 filas (Rosario cerrada + Córdoba vigente). Diego: 1 fila nueva.
SELECT cliente_sk, cliente_id, nombre, ciudad, segmento, fecha_desde, fecha_hasta, es_actual
FROM dim_cliente_scd2 ORDER BY cliente_id, fecha_desde;

--fijarse el orden en que se crea cliente_sk, porque el num 5 es la actualizacion de An cuando se muda a Córdoba;

In [0]:
%sql
-- ===== Carga del snapshot 3 (Ana a Mendoza; Carla pasa a Premium) =====
CREATE OR REPLACE TEMP VIEW stg_clientes AS
SELECT * FROM v_clientes_hash WHERE snapshot_fecha = DATE'2026-06-01';

MERGE INTO dim_cliente_scd2 AS t
USING (
  SELECT s.cliente_id AS merge_key, s.*
  FROM stg_clientes s
  UNION ALL
  --crucial para obtener los cambios individuales de fila, acá trae el de Ana y Carla
  SELECT CAST(NULL AS INT) AS merge_key, s.* 
  FROM stg_clientes s
  JOIN dim_cliente_scd2 d
    ON d.cliente_id = s.cliente_id AND d.es_actual = TRUE
  WHERE d.hash_diff <> s.hash_diff 
) AS src
  ON t.cliente_id = src.merge_key AND t.es_actual = TRUE
WHEN MATCHED AND t.hash_diff <> src.hash_diff THEN
  UPDATE SET 
  fecha_hasta = src.snapshot_fecha, 
  es_actual = FALSE
WHEN NOT MATCHED THEN
  INSERT (cliente_id, nombre, email, ciudad, provincia, segmento, hash_diff,
          fecha_desde, fecha_hasta, es_actual)
  VALUES (src.cliente_id, src.nombre, src.email, src.ciudad, src.provincia, src.segmento, src.hash_diff,
          src.snapshot_fecha, --ontroduce el fecha_hasta como el snapshot_fecha
          DATE'9999-12-31', --fecha_hasta
           TRUE); --es_actual

SELECT cliente_sk, cliente_id, nombre, ciudad, segmento, fecha_desde, fecha_hasta, es_actual
FROM dim_cliente_scd2 ORDER BY cliente_id, fecha_desde;

**Resultado esperado (7 filas):**

| cliente_id | nombre | ciudad | segmento | fecha_desde | fecha_hasta | es_actual |
|---|---|---|---|---|---|---|
| 1 | Ana Pérez | Rosario | Standard | 2026-01-01 | 2026-03-01 | FALSE |
| 1 | Ana Pérez | Córdoba | Standard | 2026-03-01 | 2026-06-01 | FALSE |
| 1 | Ana Pérez | Mendoza | Standard | 2026-06-01 | 9999-12-31 | TRUE |
| 2 | Bruno Díaz | Córdoba | Standard | 2026-01-01 | 9999-12-31 | TRUE |
| 3 | Carla Gómez | CABA | Standard | 2026-01-01 | 2026-06-01 | FALSE |
| 3 | Carla Gómez | CABA | Premium | 2026-06-01 | 9999-12-31 | TRUE |
| 4 | Diego Ruiz | Mendoza | Standard | 2026-03-01 | 9999-12-31 | TRUE |

Fijate que **la `fecha_hasta` de una versión es igual a la `fecha_desde` de la siguiente**: sin huecos y sin solapamientos.

![caso completo scd2_1795632864878.png](./caso completo scd2_1795632864878.png "caso completo scd2_1795632864878.png")

### Controles de calidad de una dimensión SCD2 (correlos siempre)
Las dos primeras consultas **tienen que devolver 0 filas**.

In [0]:
%sql
-- Control 1: no puede haber más de una versión vigente por BK
SELECT cliente_id, COUNT(*) AS vigentes
FROM dim_cliente_scd2
WHERE es_actual = TRUE
GROUP BY cliente_id
HAVING COUNT(*) > 1;

In [0]:
%sql
-- Control 2: no puede haber huecos ni solapamientos entre versiones consecutivas de un mismo BK
SELECT cliente_id, fecha_desde, fecha_hasta, fecha_hasta_anterior
FROM (
  SELECT cliente_id, fecha_desde, fecha_hasta,
         LAG(fecha_hasta) OVER (PARTITION BY cliente_id ORDER BY fecha_desde) AS fecha_hasta_anterior
  FROM dim_cliente_scd2
)
WHERE fecha_hasta_anterior IS NOT NULL AND fecha_hasta_anterior <> fecha_desde;

### Consultas típicas sobre una SCD2

In [0]:
%sql
-- (a) Versión VIGENTE de cada cliente
SELECT cliente_sk, cliente_id, nombre, ciudad, segmento
FROM dim_cliente_scd2
WHERE es_actual = TRUE
ORDER BY cliente_id;

In [0]:
%sql
-- (b) "¿Dónde vivía Ana el 2026-04-01?"  (consulta AS OF: la que Tipo 1 y Tipo 3 no pueden responder). Resultado: Córdoba
SELECT cliente_id, nombre, ciudad, fecha_desde, fecha_hasta
FROM dim_cliente_scd2
WHERE cliente_id = 1
  AND DATE'2026-04-01' >= fecha_desde
  AND DATE'2026-04-01' <  fecha_hasta;      -- semiabierto: NO usar BETWEEN

In [0]:
%sql
-- (c) Vista práctica para quien consulta Gold y solo quiere el estado actual
CREATE OR REPLACE VIEW dim_cliente_actual AS
SELECT cliente_sk, cliente_id, nombre, 
email, ciudad, provincia, segmento
FROM dim_cliente_scd2
WHERE es_actual = TRUE;

SELECT * FROM dim_cliente_actual ORDER BY cliente_id;

### Cargar la fact con la SK de la versión correcta

Acá está la clave de todo el modelo: **la fact guarda la SK de la versión vigente al momento de la venta**.

In [0]:
%sql
CREATE OR REPLACE TABLE fact_ventas_scd2 AS
SELECT
  v.venta_id,
  d.cliente_sk,            -- SK de la VERSION del cliente vigente el dia de la venta
  v.fecha_venta,
  v.monto
FROM raw_ventas v
LEFT JOIN dim_cliente_scd2 d
  ON  d.cliente_id  = v.cliente_id           -- BK
  AND v.fecha_venta >= d.fecha_desde         -- vigencia semiabierta [desde, hasta)
  AND v.fecha_venta <  d.fecha_hasta;

-- Control: ninguna venta debería quedar sin SK (si queda NULL, la venta es anterior a la 1ra versión del cliente)
SELECT COUNT(*) AS ventas, SUM(CASE WHEN cliente_sk IS NULL THEN 1 ELSE 0 END) AS sin_sk
FROM fact_ventas_scd2;

> **Sobre las ventas sin SK:** en producción no se deja `NULL`. Se crea un registro "Desconocido" en la dimensión (por ejemplo con `cliente_sk = -1`) y se usa `COALESCE(d.cliente_sk, -1)`. Para poder insertar ese `-1` a mano, la SK debe ser `GENERATED BY DEFAULT AS IDENTITY` (con `ALWAYS` Databricks no deja insertar el valor).

> **Alternativa en 2 pasos (la forma "manual" que se ve en el ejercicio 1.7 / 1.8 de la semana 5).** Hace lo mismo, pero en dos sentencias:
> ```sql
> -- Paso 1: CERRAR las versiones vigentes que cambiaron
> MERGE INTO dim_cliente_scd2 t
> USING stg_clientes s
>   ON t.cliente_id = s.cliente_id AND t.es_actual = TRUE
> WHEN MATCHED AND t.hash_diff <> s.hash_diff THEN
>   UPDATE SET fecha_hasta = s.snapshot_fecha, es_actual = FALSE;
>
> -- Paso 2: INSERTAR la versión nueva de todo cliente que ya no tiene versión vigente (cambiaron + nuevos)
> INSERT INTO dim_cliente_scd2 (cliente_id, nombre, email, ciudad, provincia, segmento, hash_diff,
>                               fecha_desde, fecha_hasta, es_actual)
> SELECT s.cliente_id, s.nombre, s.email, s.ciudad, s.provincia, s.segmento, s.hash_diff,
>        s.snapshot_fecha, DATE'9999-12-31', TRUE
> FROM stg_clientes s
> WHERE NOT EXISTS (SELECT 1 FROM dim_cliente_scd2 d
>                   WHERE d.cliente_id = s.cliente_id AND d.es_actual = TRUE);
> ```
> **Es más fácil de leer, pero el orden importa** (primero cerrar, después insertar) y si el proceso falla entre los dos pasos queda la dimensión sin versión vigente para esos clientes. La versión de un solo `MERGE` evita ese problema.

---
# 7. SCD Tipo 3: valor actual + valor anterior

**Cuándo:** solo necesito comparar el **antes y el después del último cambio** de **un atributo puntual** (por ejemplo, reportar con la ciudad nueva pero poder ver de dónde venían).

**Cómo se modela:** una columna extra por cada atributo del que quiero recordar el valor previo (`ciudad_anterior`), y opcionalmente la fecha del cambio.

**Cómo se carga:** `MERGE` por BK. Cuando cambia el atributo vigilado: `ciudad_anterior = t.ciudad` (el valor viejo, antes de pisarlo) y `ciudad = s.ciudad`. **Ojo:** la actualización de `ciudad_anterior` debe dispararse **solo cuando cambia la ciudad**, no por cualquier cambio de la fila (si no, un cambio de segmento pisaría la ciudad anterior con la actual).

In [0]:
%sql
CREATE OR REPLACE TABLE dim_cliente_scd3 ( 
  cliente_sk           BIGINT GENERATED ALWAYS AS IDENTITY,
  cliente_id           INT NOT NULL,                -- BK
  nombre               STRING,
  email                STRING,
  ciudad               STRING,                      -- valor ACTUAL
  ciudad_anterior      STRING,                      -- valor inmediatamente ANTERIOR (NULL si nunca cambió)
  fecha_cambio_ciudad  DATE,                        -- cuándo se produjo ese último cambio
  provincia            STRING,
  segmento             STRING,                      -- este atributo se trata como Tipo 1 (se pisa)
  hash_diff            STRING
) USING DELTA
COMMENT 'Dimension cliente - SCD Tipo 3 (ciudad actual + ciudad anterior)';

In [0]:
%sql
-- ===== Carga del snapshot 1 (inicial): ciudad_anterior queda en NULL =====
CREATE OR REPLACE TEMP VIEW stg_clientes AS
SELECT * FROM v_clientes_hash WHERE snapshot_fecha = DATE'2026-01-01';

MERGE INTO dim_cliente_scd3 AS t
USING stg_clientes AS s
  ON t.cliente_id = s.cliente_id
-- (1) Cambió la CIUDAD: guardo la anterior y pongo la nueva (<=> es igualdad que tolera NULL)
WHEN MATCHED AND NOT (t.ciudad <=> s.ciudad) THEN
  UPDATE SET ciudad_anterior = t.ciudad,                 -- valor viejo (antes del UPDATE)
             ciudad = s.ciudad,                          -- valor nuevo
             fecha_cambio_ciudad = s.snapshot_fecha,
             nombre = s.nombre, email = s.email, provincia = s.provincia,
             segmento = s.segmento, hash_diff = s.hash_diff
-- (2) Cambió otra cosa pero NO la ciudad: pisar el resto SIN tocar ciudad_anterior
WHEN MATCHED AND t.hash_diff <> s.hash_diff THEN
  UPDATE SET nombre = s.nombre, email = s.email, provincia = s.provincia,
             segmento = s.segmento, hash_diff = s.hash_diff
WHEN NOT MATCHED THEN
  INSERT (cliente_id, nombre, email, ciudad, ciudad_anterior, fecha_cambio_ciudad, provincia, segmento, hash_diff)
  VALUES (s.cliente_id, s.nombre, s.email, s.ciudad, NULL, NULL, s.provincia, s.segmento, s.hash_diff);

SELECT cliente_id, nombre, ciudad, ciudad_anterior, fecha_cambio_ciudad, segmento
FROM dim_cliente_scd3 ORDER BY cliente_id;

In [0]:
%sql
-- ===== Carga del snapshot 2: Ana pasa a Córdoba -> ciudad_anterior = Rosario =====
CREATE OR REPLACE TEMP VIEW stg_clientes AS
SELECT * FROM v_clientes_hash WHERE snapshot_fecha = DATE'2026-03-01';

MERGE INTO dim_cliente_scd3 AS t
USING stg_clientes AS s
  ON t.cliente_id = s.cliente_id
WHEN MATCHED AND NOT (t.ciudad <=> s.ciudad) THEN
  UPDATE SET ciudad_anterior = t.ciudad, ciudad = s.ciudad, fecha_cambio_ciudad = s.snapshot_fecha,
             nombre = s.nombre, email = s.email, provincia = s.provincia,
             segmento = s.segmento, hash_diff = s.hash_diff
WHEN MATCHED AND t.hash_diff <> s.hash_diff THEN
  UPDATE SET nombre = s.nombre, email = s.email, provincia = s.provincia,
             segmento = s.segmento, hash_diff = s.hash_diff
WHEN NOT MATCHED THEN
  INSERT (cliente_id, nombre, email, ciudad, ciudad_anterior, fecha_cambio_ciudad, provincia, segmento, hash_diff)
  VALUES (s.cliente_id, s.nombre, s.email, s.ciudad, NULL, NULL, s.provincia, s.segmento, s.hash_diff);

SELECT cliente_id, nombre, ciudad, ciudad_anterior, fecha_cambio_ciudad, segmento
FROM dim_cliente_scd3 ORDER BY cliente_id;

In [0]:
%sql
-- ===== Carga del snapshot 3: Ana pasa a Mendoza -> ciudad_anterior = Córdoba (Rosario se PIERDE) =====
-- Carla cambia solo de segmento -> entra por la cláusula (2) y su ciudad_anterior no se toca
CREATE OR REPLACE TEMP VIEW stg_clientes AS
SELECT * FROM v_clientes_hash WHERE snapshot_fecha = DATE'2026-06-01';

MERGE INTO dim_cliente_scd3 AS t
USING stg_clientes AS s
  ON t.cliente_id = s.cliente_id
WHEN MATCHED AND NOT (t.ciudad <=> s.ciudad) THEN
  UPDATE SET ciudad_anterior = t.ciudad, ciudad = s.ciudad, fecha_cambio_ciudad = s.snapshot_fecha,
             nombre = s.nombre, email = s.email, provincia = s.provincia,
             segmento = s.segmento, hash_diff = s.hash_diff
WHEN MATCHED AND t.hash_diff <> s.hash_diff THEN
  UPDATE SET nombre = s.nombre, email = s.email, provincia = s.provincia,
             segmento = s.segmento, hash_diff = s.hash_diff
WHEN NOT MATCHED THEN
  INSERT (cliente_id, nombre, email, ciudad, ciudad_anterior, fecha_cambio_ciudad, provincia, segmento, hash_diff)
  VALUES (s.cliente_id, s.nombre, s.email, s.ciudad, NULL, NULL, s.provincia, s.segmento, s.hash_diff);

SELECT cliente_id, nombre, ciudad, ciudad_anterior, fecha_cambio_ciudad, segmento
FROM dim_cliente_scd3 ORDER BY cliente_id;

**Resultado esperado (4 filas):**

| cliente_id | nombre | ciudad | ciudad_anterior | fecha_cambio_ciudad | segmento |
|---|---|---|---|---|---|
| 1 | Ana Pérez | Mendoza | **Córdoba** | 2026-06-01 | Standard |
| 2 | Bruno Díaz | Córdoba | NULL | NULL | Standard |
| 3 | Carla Gómez | CABA | NULL | NULL | **Premium** |
| 4 | Diego Ruiz | Mendoza | NULL | NULL | Standard |

**Qué mirar:**
- **Rosario desapareció** de la historia de Ana: Tipo 3 solo conserva un cambio hacia atrás.
- **El segmento de Carla se pisó** (comportamiento Tipo 1): Tipo 3 solo recuerda el atributo para el que le armaste columna `_anterior`. Para recordar también el segmento anterior habría que agregar `segmento_anterior`.
- **Tipo 3 crece en columnas, no en filas.** Una columna `_anterior` por cada atributo que quiero recordar. Si necesito más de un cambio hacia atrás, Tipo 3 no alcanza: es Tipo 2.

---
# 8. Comparación con la fact: ¿qué cambia en los reportes?

Las mismas 7 ventas (total 9.200) agrupadas por **ciudad del cliente**, con cada tipo de dimensión.

In [0]:
%sql
-- Ventas por ciudad: Tipo 1 y Tipo 3 (columna ciudad actual) vs Tipo 2 (ciudad al momento de la venta)
WITH t1_t3 AS (   -- en Tipo 1 y Tipo 3 el SK no cambia: se une por BK, sin fechas
  SELECT d.ciudad, SUM(v.monto) AS ventas
  FROM raw_ventas v
  JOIN dim_cliente_scd1 d ON d.cliente_id = v.cliente_id
  GROUP BY d.ciudad
),
t2 AS (           -- en Tipo 2 se une por SK (la fact ya trae la SK de la versión correcta)
  SELECT d.ciudad, SUM(f.monto) AS ventas
  FROM fact_ventas_scd2 f
  JOIN dim_cliente_scd2 d ON d.cliente_sk = f.cliente_sk
  GROUP BY d.ciudad
)
SELECT ciudad,
       t1_t3.ventas AS ventas_tipo1_y_tipo3,
       t2.ventas    AS ventas_tipo2
FROM t1_t3
FULL OUTER JOIN t2 USING (ciudad)
ORDER BY ciudad;

**Resultado esperado:**

| ciudad | ventas_tipo1_y_tipo3 | ventas_tipo2 |
|---|---|---|
| CABA | 2000 | 2000 |
| Córdoba | 500 | 2500 |
| Mendoza | 6700 | 3700 |
| Rosario | NULL | 1000 |

- **Tipo 1 y Tipo 3** mandan todas las ventas de Ana (6.000) a **Mendoza**, aunque dos de las tres se hicieron cuando vivía en otro lado. Rosario ni aparece en el reporte.
- **Tipo 2** reparte las ventas de Ana donde corresponde: Rosario 1.000, Córdoba 2.000, Mendoza 3.000.
- Los dos suman **9.200**: cambia el reparto, no el total.

**¿Cuál es el "correcto"?** Depende de la pregunta de negocio. Si pregunto *"¿cuánto vendimos en cada ciudad cuando se vendió?"* es Tipo 2. Si pregunto *"¿cuánto vendimos a los clientes que hoy viven en cada ciudad?"* es Tipo 1 (o la ciudad actual de una Tipo 2 con `es_actual`). **Por eso el tipo de SCD no es una decisión técnica: es una decisión de negocio sobre la pregunta que el reporte tiene que responder.**

In [0]:
%sql
-- El ERROR CLÁSICO: unir la fact a una SCD2 por BK, sin mirar fechas ni SK.
-- Cada venta se repite tantas veces como versiones tenga el cliente.
SELECT SUM(v.monto) AS total_inflado          -- debería dar 9200
FROM raw_ventas v
JOIN dim_cliente_scd2 d ON d.cliente_id = v.cliente_id;

**Resultado: 23.200 en lugar de 9.200.** Ana tiene 3 versiones (sus 6.000 se cuentan 3 veces = 18.000), Carla tiene 2 (2.000 x 2 = 4.000), Bruno 1 (500) y Diego 1 (700): 18.000 + 4.000 + 500 + 700 = 23.200.
Moraleja: con SCD2, la fact **siempre** une por SK, y la SK se resuelve **una vez**, al cargar la fact, con el lookup por BK + fecha.

---
# 9. Errores clásicos y checklist

## 9.1 Errores que se repiten

| # | Error | Consecuencia | Cómo evitarlo |
|---|---|---|---|
| 1 | Unir la fact a una SCD2 por BK | Ventas duplicadas, totales inflados | Fact con la SK de la versión; lookup por BK + fecha al cargar (sección 6) |
| 2 | Usar `BETWEEN` con `fecha_hasta` = `fecha_desde` de la siguiente | El día del cambio cae en las 2 versiones | Intervalo semiabierto: `>= fecha_desde AND < fecha_hasta` |
| 3 | Comparar columnas con `<>` o `!=` cuando pueden ser `NULL` | `NULL <> 'x'` da `NULL` (no `TRUE`): el cambio no se detecta | Usar `hash_diff` con `COALESCE`, o `NOT (a <=> b)` |
| 4 | Meter en el hash columnas que cambian todo el tiempo (`updated_at`, `ultimo_login`) | Una versión nueva en cada carga | Hash solo con atributos que importan al historial |
| 5 | En SCD3, actualizar `*_anterior` ante cualquier cambio de la fila | El valor anterior se pierde por un cambio ajeno | Cláusulas separadas: una por cambio del atributo vigilado, otra para el resto (sección 7) |
| 6 | Insertar la versión nueva **antes** de cerrar la vieja (con `UPDATE` + `INSERT` sueltos) | El `UPDATE` cierra también la versión recién insertada | Un solo `MERGE` con `merge_key`, o cerrar siempre primero |
| 7 | Staging con **más de una fila por BK** | El `MERGE` falla (varias filas de origen matchean la misma fila destino) o genera versiones basura | Deduplicar en Silver y verificar que el staging tenga 1 fila por BK |
| 8 | Cargar snapshots en desorden | Intervalos incoherentes | Procesar siempre en orden cronológico |
| 9 | Aplicar SCD2 a todas las dimensiones | Tablas enormes, joins más complejos, ningún beneficio | Aplicar el árbol de decisión de la sección 2.3 |
| 10 | Olvidar los clientes que **desaparecen** del origen | La versión vigente queda abierta para siempre | Decidir el criterio de baja (por ejemplo cerrar la versión con `WHEN NOT MATCHED BY SOURCE`, o una columna `eliminado`) |

## 9.2 Checklist al diseñar una dimensión en Gold

- [ ] ¿Cuál es la **BK**? ¿Es única en el staging (1 fila por BK)?
- [ ] ¿La dimensión tiene **SK** (`IDENTITY`) y la fact apunta a la SK, no a la BK?
- [ ] Para cada atributo que cambia: ¿es corrección (Tipo 1), necesito historia (Tipo 2) o solo el anterior (Tipo 3)? **Anotar la decisión y el motivo.**
- [ ] Si hay Tipo 2: `fecha_desde`, `fecha_hasta`, `es_actual`, `hash_diff`; intervalo semiabierto; qué columnas entran al hash.
- [ ] ¿De dónde sale la **fecha efectiva** del cambio? (fecha del snapshot, fecha del evento en la fuente, o fecha de carga)
- [ ] ¿Cómo se cargan las **filas nuevas, modificadas y eliminadas**?
- [ ] ¿La fact resuelve la SK con el lookup por fecha? ¿Qué pasa con las ventas sin versión? (registro "Desconocido")
- [ ] ¿La carga es **idempotente**? (correrla dos veces con los mismos datos no cambia nada)
- [ ] Controles de calidad: una sola versión vigente por BK, sin huecos ni solapamientos.

---
# 10. Otros tipos y atajos (para reconocerlos)

| Tipo | Idea |
|---|---|
| **Tipo 0** | El atributo **nunca** cambia (fecha de alta, fecha de nacimiento). Se carga una vez y no se actualiza. |
| **Tipo 4** | La dimensión guarda solo el estado actual y el historial va a una **tabla aparte** (`dim_cliente_historial`). |
| **Tipo 6** | Híbrido de 1 + 2 + 3: filas versionadas (Tipo 2) y además una columna con el valor **actual** repetido en todas las versiones. Permite reportar "con la ciudad de su época" y "con la ciudad de hoy" desde la misma tabla. |
| **Híbrido por columna** | Muy común en la práctica: en la misma dimensión, algunos atributos son Tipo 2 (los que van al `hash_diff`) y otros Tipo 1 (se pisan en todas las versiones). |

**Atajo en Databricks:** existe una forma declarativa de mantener SCD1/SCD2 sin escribir el `MERGE` a mano (en Lakeflow Spark Declarative Pipelines, con `AUTO CDC`, antes `APPLY CHANGES`). Conviene conocer el `MERGE` a mano primero, porque es lo que esa herramienta hace por debajo. Para el detalle exacto de sintaxis, la documentación oficial de Databricks es la fuente.

In [0]:
%sql
-- (Opcional) Limpieza: borra todo lo creado por este notebook
-- DROP SCHEMA IF EXISTS bootcamp.demo_scd CASCADE;